# MLX Graph Model and Scikit-Learn
There may be times that typical graph models may not produce sufficiently performing models. One culprit is the size of the dataset is too small to be effectively utilized by deep learning methods. To combat this, graph features can be utilized as the feature vector as input for traditional machine learning (ML) models from Scikit-Learn. There have been several examples of this by researchers ([here](https://www.nature.com/articles/s42256-024-00923-6), [here](https://github.com/Chokyotager/NotYetAnotherNightshade/tree/main), and [here](https://jonswain.github.io/ai/cheminformatics/data%20science/machine%20learning/2025/09/12/chemprop-rf.html
)). To do this, the molecules will be converted into molecular graphs. Then the graph latent space will be exported. These features are in numpy format, which can then be used to build traditional ML models.

This tutorial will demo how to create molecular graph features and output the latent space using MLX. This will then be used to build a RandomForrest classifier model and predict sample molecules.

## Import packages
The MolMiner comes with four graph models - Graph Convolutional Neural Network (GCN), Graph Isomorphism Network (GINE), Graph Sage (GSAGE), and Graph Attention Networks (GAT). These classes can be imported from the **latent_space** module.

In [1]:
import pandas as pd
from tqdm import tqdm
from mminer_graphs.mlx_models import MolToGraph, EarlyStopping
from mminer_graphs.mlx_models.mlx_latent_space import GCN

## Dataset Prep
Similar to the other tutorials, the bace dataset will be used.

In [2]:
data = pd.read_csv("../dataset/bace.csv")
cols = ['CID', 'smiles', 'Class', 'Model']
dataset = data[cols]
print(f"The shape of the dataset: {dataset.shape}")
dataset.head()

The shape of the dataset: (1513, 4)


,CID,smiles,Class,Model
0,BACE_1,O1CC[C@@H](NC(=O)[C@@H](Cc2cc3cc(ccc3nc2N)-c2c...,1,train
1,BACE_2,Fc1cc(cc(F)c1)C[C@H](NC(=O)[C@@H](N1CC[C@](NC(...,1,train
2,BACE_3,S1(=O)(=O)N(c2cc(cc3c2n(cc3CC)CC1)C(=O)N[C@H](...,1,train
3,BACE_4,S1(=O)(=O)C[C@@H](Cc2cc(O[C@H](COCC)C(F)(F)F)c...,1,train
4,BACE_5,S1(=O)(=O)N(c2cc(cc3c2n(cc3CC)CC1)C(=O)N[C@H](...,1,train


The dataset is separated by train/test labels. The molecule smiles string and class label will be extracted from the table and, in a list, will be converted into molecular graph data. The first five entries in train_graphs will be printed to quickly visualize the data.

In [3]:
train = dataset[dataset.iloc[:, -1] == 'train']
test = dataset[dataset.iloc[:, -1] == 'test']

train_graphs = [MolToGraph(smi=smi, label=y).graph_data for smi, y in tqdm(zip(train.smiles, train.Class), desc="Converting Train Graphs")]
test_graphs = [MolToGraph(smi=smi, label=y).graph_data for smi, y in tqdm(zip(test.smiles, test.Class), desc="Converting Test Graphs")]

train_graphs[:5]

Converting Train Graphs: 1376it [00:02, 615.43it/s]
Converting Test Graphs: 137it [00:00, 611.76it/s]


[GraphData(
 	edge_index(shape=(2, 70), int32)
 	node_features(shape=(32, 142), float32)
 	edge_features(shape=(70, 16), float32)
 	y(shape=(1,), float32)),
 GraphData(
 	edge_index(shape=(2, 100), int32)
 	node_features(shape=(47, 142), float32)
 	edge_features(shape=(100, 16), float32)
 	y(shape=(1,), float32)),
 GraphData(
 	edge_index(shape=(2, 92), int32)
 	node_features(shape=(42, 142), float32)
 	edge_features(shape=(92, 16), float32)
 	y(shape=(1,), float32)),
 GraphData(
 	edge_index(shape=(2, 84), int32)
 	node_features(shape=(40, 142), float32)
 	edge_features(shape=(84, 16), float32)
 	y(shape=(1,), float32)),
 GraphData(
 	edge_index(shape=(2, 96), int32)
 	node_features(shape=(44, 142), float32)
 	edge_features(shape=(96, 16), float32)
 	y(shape=(1,), float32))]

## GCN
I this example, the GCN model will be used. By dfault, it will utilize AtomGraph features. Printing the model will show the graph model structure. In this example, three graph layers are used with an output of 128 dimensions.

In [4]:
graph_model = GCN(node_dim=142, hidden_dim=128, graph_layers=3, reduce_graph=False)
graph_model

GCN(
  (gcn_conv.0): GCNConv(
    (linear): Linear(input_dims=142, output_dims=128, bias=True)
  )
  (gcn_conv.1): GCNConv(
    (linear): Linear(input_dims=128, output_dims=128, bias=True)
  )
  (gcn_conv.2): GCNConv(
    (linear): Linear(input_dims=128, output_dims=128, bias=True)
  )
)

### Extract Train/Test Data

Once the graph model has been created, the graph latent space can be extracted. This is done using the **split_graph_space** and will return train_x, train_y, test_x, and test_y data. This is helpful as it can be used to feed data directly into sklearn models.

In [5]:
train_x, train_y, test_x, test_y = graph_model.split_graph_space(train_graphs, test_graphs)

Converting training graph data: 1376it [00:03, 379.62it/s]
Converting test graph data: 137it [00:00, 381.46it/s]


Visualizing the latent space vector gives the graph latent space as a vector matching the final hidden dimensions. In this case, the vector for the training data is used.

In [6]:
train_x

array([[0.0000000e+00, 0.0000000e+00, 0.0000000e+00, ..., 6.9696344e-02,
        3.0501044e-05, 6.9641061e-02],
       [0.0000000e+00, 0.0000000e+00, 0.0000000e+00, ..., 6.8501495e-02,
        2.2279697e-04, 6.4883888e-02],
       [0.0000000e+00, 0.0000000e+00, 0.0000000e+00, ..., 7.0891663e-02,
        2.3548782e-04, 7.2967812e-02],
       ...,
       [0.0000000e+00, 0.0000000e+00, 0.0000000e+00, ..., 7.2221301e-02,
        3.3204462e-05, 7.6051377e-02],
       [0.0000000e+00, 0.0000000e+00, 0.0000000e+00, ..., 7.4024059e-02,
        0.0000000e+00, 8.1136949e-02],
       [0.0000000e+00, 0.0000000e+00, 0.0000000e+00, ..., 6.8758667e-02,
        3.0042132e-05, 6.8128213e-02]], shape=(1376, 128), dtype=float32)

## Random Forest Model

Once the graph latent space is generated, the arrays can be used to fit an sklearn model. In this case, a Random Forest Classifier.

In [7]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(train_x, train_y)

y_pred = rf.predict(test_x)
print("Accuracy:", accuracy_score(test_y, y_pred))

Accuracy: 0.8029197080291971


## Making Predictions

Making predictions requires a similar workflow - converting smiles string into graph data, obtaining the graph latent space, and using the fitted sklearn model for prediction.

**NOTE:** The molecules used for this example were taken from the training dataset above. Thus, the model has "cheated" and seen this data before. Future testing should be done using unseen data. 

In [8]:
# prediction
to_predict = ["O1CC[C@@H](NC(=O)[C@@H](Cc2cc3cc(ccc3nc2N)-c2ccccc2C)C)CC1(C)C", "Fc1cc(cc(F)c1)C[C@H](NC(=O)[C@@H](N1CC[C@](NC(=O)C)(CC(C)C)C1=O)CCc1ccccc1)[C@H](O)[C@@H]1[NH2+]C[C@H](OCCC)C1", "FC(F)(F)Oc1ccc(cc1)C1(N=C(N)N(C)C1=O)c1cc(ccc1)-c1ccc(OC)nc1"]

data = [MolToGraph(smi=smi).graph_data for smi in tqdm(to_predict, desc="Converting Data to Graphs")]

graph_latent_space = graph_model.extract_graph_space(data)
graph_latent_space

Converting Data to Graphs: 100%|██████████| 3/3 [00:00<00:00, 433.91it/s]
Converting graph data: 3it [00:00, 306.32it/s]


array([[0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        2.84385472e-03, 0.00000000e+00, 5.89537658e-02, 0.00000000e+00,
        3.16166668e-04, 5.46773663e-03, 0.00000000e+00, 2.49051303e-02,
        0.00000000e+00, 9.00652446e-03, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        3.66321281e-02, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 2.77788960e-04, 0.00000000e+00, 8.72410163e-02,
        0.00000000e+00, 0.00000000e+00, 8.27285927e-03, 0.00000000e+00,
        1.16853051e-01, 3.56918899e-04, 0.00000000e+00, 0.00000000e+00,
        2.82854284e-03, 6.54564127e-02, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 1.66749563e-02, 1.70193089e-03, 0.00000000e+00,
        0.00000000e+00, 2.32509244e-02, 0.00000000e+00, 0.00000000e+00,
        1.66794583e-02, 0.00000000e+00, 7.63222501e-02, 5.58338910e-02,
        8.38405685e-04, 0.00000000e+00, 6.99776560e-02, 1.037584

In [9]:
predictions = rf.predict(graph_latent_space)
predictions

array([1., 1., 0.], dtype=float32)

Optionally, users can also output the results as probability.

In [10]:
predictions = rf.predict_proba(graph_latent_space)
predictions

array([[0.15, 0.85],
       [0.31, 0.69],
       [0.74, 0.26]])